In [1]:
import math
import numpy as np 
import pandas as pd
from collections import defaultdict

In [16]:
class XGBoostModel():

    def __init__(self, params, random_seed=None):
        # self.params = defaultdict(params)
        self.params = params
        self.subsample = self.params['subsample'] \
            if self.params['subsample'] else 0.1
        self.learning_rate = self.params['learning_rate'] \
            if self.params['learning_rate'] else 0.3
        self.base_prediction = self.params['base_score'] \
            if self.params['base_score'] else 0.5
        self.max_depth = self.params['max_depth'] \
            if self.params['max_depth'] else 5

        self.rng = np.random.default_rng(seed=random_seed)
        

In contrast to boosting in the classic GBM, instead of computing residuals between the current predictions and the target, we compute gradients and hessians of the loss function with respect to the current predictions, and instead of predicting residuals with a decision tree, we fit a special XGBoost tree booster, using the gradients and hessians

In [34]:
def fit(self,X, y, objective, n_estimators):
    current_prediction = self.base_prediction*np.ones(shape=y.shape) #Starting mein model will produce same prediction. Then we will generate gradient and hessian and the work on it.
    self.models = []
    sample_idx = 0

    for i in range(n_estimators):
        gradients = objective.gradient(y,current_prediction) #current prediction, actual se kitna glt hain.
        hessians = objective.hessian(y, current_prediction) #rate change of the loss function

        if self.subsample == 1:
            sample_idx= None
        else: 
            sample_idx = self.rng.choice(len(y), size=math.floor(self.subsample*len(y)), replace = False)   #subsample*len(y) means total y me se kitne samples select krne hain. 
            #self.rng.choice-> randomly select the samples 

        Tree  = TreeBooster( X, gradients, hessians, self.params, self.max_depth, sample_idx) #create new trees

        current_prediction += self.learning_rate *Tree.predict(X)  #update the predictions 

        self.models.append(Tree)

def predict(self, X): 
    return (
        self.base_prediction + self.learning_rate * np.sum([tree.predict(X) for tree in self.models], axis = 0)
    )

XGBoostModel.fit = fit
XGBoostModel.predict = predict 

$$
\text{Base Prediction} = \text{Base Prediction} + \eta \sum_{k=1}^{K} \text{Tree}_k(X)
$$

Now we recursively build a binary tree structure by finding the best split rule for each node in the tree. The main difference is the criterion for evaluating splits and the way that we define a leaf's predicted value. Instead of being functions of the target values of the instances in each node, the criterion and predicted values are functions of the instance gradients and hessians

In [26]:
class TreeBooster():
    def __init__ (self, X, g, h, params, max_depth, idx = None):

        self.params = params
        self.max_depth = max_depth 
        self.min_child_weight = params['min_child_weight'] \
            if params['min_child_weight'] else 1.0 #Ye basically check karta hai ki child node mein enough Hessian weight hai ya nahi.
        self.reg_lambda = params['reg_lambda'] \
            if params['reg_lambda'] else 1.0 # for L2 regularization, lambda. Controls the overfitting
        self.gamma = params['gamma'] \
        if params['gamma'] else 0.0 #mini improvement required to initiate split 
        self.colsample_bynode = params['colsample_bynode'] \
        if params['colsample_bynode'] else 1.0 # Har node par kitne features consider karne hain.
        

        if isinstance(g, pd.Series):
            g = g.values  # Agar g aur h pandas Series hain, toh convert them into numpy array
        if isinstance(h, pd.Series):
            h = h.values

        if idx is None :  #Agar koi idxs nahi diya gaya, iska matlab ye root node hai.
            idx = np.arange(len(g)) # if len(g) == 5, idx = [0,1,2,3,4]. Root node contains all the samples

        self.X, self.h, self.g, self.idx = X, h,g,idx

        self.n = len(idx) #current node me kitne samples hain 
        self.c = X.shape[1] #total number of features/columns

        # if, X.shape = (100, 5); 100 samples, 5 features

        #Calculating optimal weight of the leaf 
        self.value = -g[idx].sum() / (h[idx].sum() + self.reg_lambda)

        self.best_score = 0 #it will track if we found any best score or not

        if self.max_depth> 0:  #agar accha split mil gaya toh tree ko grow karo
            self.insert_child_node()

    #splitting logic 

    def insert_child_node(self):

        for i in range(self.c):
            self.find_better_split(i) 

        if self.is_leaf: #agar best split nahi mila = leaf node reached
            return 

        x = self.X.values[self.idx,self.split_feature_idx] #us index se lekar us feature tak ke index tak data divide krna. 
        left_leaf = np.nonzero(x<self.threshold)[0] #np.nonzero returns a tupple. for ex (array([0, 1]),).. [0] make sure that we have [0,1] in our variable
        right_leaf = np.nonzero(x>= self.threshold)[0]

        self.left= TreeBooster(self.X, self.g, self.h, self.params, self.max_depth-1, self.idx[left_leaf])
        self.right= TreeBooster(self.X, self.g, self.h, self.params, self.max_depth-1, self.idx[right_leaf])
        #             Root 
        #           /      \
        #       Left        Right
        #      /    \       /    \
        #    LL     LR     RL     RR

        #(g,h,idx,value for4 every node) 

    @property
    def is_leaf(self):
        return self.best_score == 0

    #function which decides on which threshold and feature should we split our tree
    def find_better_split(self, feature_idx):
        x = self.X.values[self.idx, feature_idx] #self.idxs ensure karta hai ki sirf current node ke samples liye ja rahe hain.
        g,h = self.g[self.idx] , self.h[self.idx]

        sort_idx = np.argsort(x) #sort the values of features. sort_idx stores the indexes
        sort_g = g[sort_idx]
        sort_h = h[sort_idx]
        sort_x = x[sort_idx]

        sum_g, sum_h = g.sum(), h.sum() #we got the sum of hessians and gradients of the node we are working with 
        sum_g_right, sum_h_right = sum_g, sum_h # initially we haven't performed splitting yet, so we store all the values in right side. 
        sum_g_left, sum_h_left = 0., 0.
        for i in range(0, self.n-1):
            g_i, h_i,x_i,x_next = sort_g[i], sort_h[i], sort_x[i], sort_x[i+1] 
        # if i =1 x_i = 25, x_next = 30. potential threshold = 25 < threshold < x_next. then we can calculate the threshold by (25+30)/2 = 27.5. So we split if x<=27.5 left else: x>27.5 move right 

            #Har iteration mein ek sample Right se Left mein move hota hai.
            sum_g_left += g_i
            sum_h_left += h_i
            sum_g_right -= g_i 
            sum_h_right -= h_i 

            # Left | Right
            # -----|--------------------
            #      | 20 25 30 35 50

            # After first:

            # Left | Right
            # 20   | 25 30 35 50


            # After second:

            # 20 25 | 30 35 50

            if sum_h_left < self.min_child_weight or x_i == x_next: # we are checking if we have enough hessian or not, if not then reject the split find new threshold, also if we have same values consecutively then split is meaningless
                continue 

            if sum_h_right < self.min_child_weight: 
                #Agar right side ka Hessian minimum requirement se kam ho gaya: 
                # → aur samples left mein move karoge
                # → right aur chhota hoga
                # → future splits bhi invalid honge

                break #hence we break

            #assuming we found a splitting features, now we will see the gain value.. weather the loss is reducing or not

            gain = 0.5 * (
                (sum_g_left**2 / (sum_h_left + self.reg_lambda))
                +
                (sum_g_right**2 / (sum_h_right + self.reg_lambda))
                -
                (sum_g**2 / (sum_h + self.reg_lambda))
            ) - self.gamma/2 #gamma penalize the split. high gamma->simple tree, less overfitting, difficulty in splitting. low gamma -> easy split, complex tree


            if gain> self.best_score:
                self.split_feature_idx = feature_idx #best feature
                self.best_score = gain # Best gain 
                self.threshold = (x_i+x_next) / 2  #Best threshold 


    TreeBooster.find_better_split = find_better_split

self.value = -g[idx].sum() / (h[idx] + self.reg_lambda) is actually the formula below

$$
w^* = -\frac{\sum g_i}{\sum h_i + \lambda}
$$

g = gradient

h = hessian

λ = regularization

w* = leaf prediction/update



gain = 0.5 * ( (sum_g_left**2 / (sum_h_left + self.reg_lambda)) + (sum_g_right**2 / (sum_h_right + self.reg_lambda)) - (sum_g**2 / (sum_h + self.reg_lambda))) - self.gamma/2 is actually the formula below 


$$
\text{Gain}
=
\frac{1}{2}
\left[
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{G^2}{H+\lambda}
\right]
-
\frac{\gamma}{2}
$$


G_L = left gradients ka sum, 
H_L = left hessians ka sum

G_R = right gradients ka sum, 
H_R = right hessians ka sum

G = parent gradients ka sum, 
H = parent hessians ka sum

λ = reg_lambda, 
γ = gamma

Splitting process
```text
Features
   ↓
Candidate Thresholds
   ↓
Gradient + Hessian
   ↓
Check `min_child_weight`
   ↓
Calculate Gain
   ↓
Apply `gamma` + Regularization
   ↓
Choose Best Split
   ↓
Check `max_depth`
   ↓
Split / Leaf

colsample_bynode → Which features?
threshold → Where to split?
gradient + hessian → How good is the split?
min_child_weight + gamma → Should we allow the split?
max_depth → How deep can we go?

In [ ]:
def predict(self, X):
    return np.array([self._predict_row(row) for i, row in X.iterrows()])

def _predict_row(self, row):
    if self.is_leaf: 
        return self.value
    child = self.left if row[self.split_feature_idx] <= self.threshold \
        else self.right
    return child._predict_row(row)

TreeBooster.predict = predict 
TreeBooster._predict_row = _predict_row 

TESTING ON A DATASET

In [28]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
    
X, y = fetch_california_housing(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, 
                                                    random_state=43)

In [29]:
class SquaredErrorObjective():
    def loss(self, y, pred): return np.mean((y - pred)**2)
    def gradient(self, y, pred): return pred - y
    def hessian(self, y, pred): return np.ones(len(y))

In [40]:
import xgboost as xgb

params = {
    'learning_rate': 0.1,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bynode': 1.0,
    'reg_lambda': 1.5,
    'gamma': 0.0,
    'min_child_weight': 25,
    'base_score': 0.5,
    'tree_method': 'exact',
}
num_boost_round = 50

# train the from-scratch XGBoost model
model_scratch = XGBoostModel(params, random_seed=42)
model_scratch.fit(X_train, y_train, SquaredErrorObjective(), num_boost_round)

# train the library XGBoost model
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)
model_xgb = xgb.train(params, dtrain, num_boost_round)

C:\Users\MSII\AppData\Local\Temp\ipykernel_4380\456398594.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  child = self.left if row[self.split_feature_idx] <= self.threshold \


In [41]:
pred_scratch = model_scratch.predict(X_test)
pred_xgb = model_xgb.predict(dtest)
print(f'scratch score: {SquaredErrorObjective().loss(y_test, pred_scratch)}')
print(f'xgboost score: {SquaredErrorObjective().loss(y_test, pred_xgb)}')

C:\Users\MSII\AppData\Local\Temp\ipykernel_4380\456398594.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  child = self.left if row[self.split_feature_idx] <= self.threshold \


scratch score: 0.2434125759558149
xgboost score: 0.2432306663531763
